# 📏 Feature Scaling
**One-line description:** Normalize your features so distance-based and gradient-descent models converge faster and perform better.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/03_feature_scaling.ipynb)


In [ ]:
# Install required libraries (run this cell first in Google Colab)
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet


## 📖 What is Feature Scaling?

Feature scaling transforms numeric features so they occupy a **similar range of values**. Without scaling, features with large magnitudes dominate models that use distances or gradients.

**Analogy:** Imagine comparing a house's "number of bedrooms" (range: 1–10) against its "lot size in square feet" (range: 500–50,000). If you compute Euclidean distance, lot size completely drowns out bedrooms. Scaling puts both features on an equal footing.

### Key Scalers:
| Scaler | Formula | Output Range | When to Use |
|--------|---------|-------------|-------------|
| **StandardScaler** | (x - μ) / σ | Mean=0, Std=1 | General purpose, Gaussian-ish data |
| **MinMaxScaler** | (x - min) / (max - min) | [0, 1] | When bounded range required (neural nets) |
| **RobustScaler** | (x - median) / IQR | Robust to outliers | Data with significant outliers |
| **MaxAbsScaler** | x / max(|x|) | [-1, 1] | Sparse data, already centered at 0 |
| **Normalizer** | x / ||x|| | Unit norm per sample | Text/NLP, row-wise normalization |


## 💡 Why Does It Matter?

- **KNN**: uses Euclidean distance — large-scale features dominate
- **SVM**: kernel functions are distance-based
- **Gradient Descent** (Logistic Regression, Neural Networks): unscaled features cause slow, oscillating convergence
- **PCA**: variance-based — large-scale features dominate principal components
- **Tree models** (Random Forest, XGBoost): NOT affected — they use thresholds, not distances


## ⚙️ How Does It Work?

1. **Fit** the scaler on the **training set** (learn parameters like mean, std, min, max)
2. **Transform** both training and test sets using fitted parameters
3. **Never fit on test data** — this would cause data leakage


## 🛠️ Hands-on Code

### Step 1: Create a Housing Price dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   MaxAbsScaler, Normalizer)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 1000

# Housing dataset with features of vastly different scales
df = pd.DataFrame({
    'LotArea':         np.random.normal(10000, 3000, n).clip(2000, 25000),   # sq ft: hundreds-thousands
    'LotFrontage':     np.random.normal(70, 20, n).clip(20, 150),             # feet: tens
    'OverallQual':     np.random.randint(1, 11, n).astype(float),             # scale 1-10
    'YearBuilt':       np.random.randint(1900, 2023, n).astype(float),        # years: thousands
    'GrLivArea':       np.random.normal(1500, 500, n).clip(400, 4000),        # sq ft: hundreds-thousands
    'Bedrooms':        np.random.randint(1, 6, n).astype(float),              # count: 1-5
    'Bathrooms':       np.random.choice([1, 1.5, 2, 2.5, 3, 3.5], n),        # count: 1-3.5
    'GarageArea':      np.random.normal(480, 180, n).clip(0, 1200),           # sq ft: 0-1200
    'TotalBsmtSF':     np.random.normal(1000, 350, n).clip(0, 3000),          # sq ft
    'PoolArea':        np.random.exponential(10, n).clip(0, 500),             # sq ft, heavily skewed
})

# Add some realistic outliers in PoolArea (like real estate data)
outlier_idx = np.random.choice(n, 20, replace=False)
df.loc[outlier_idx, 'PoolArea'] = np.random.uniform(400, 800, 20)

# Target: House price (correlated with features)
df['SalePrice'] = (
    df['GrLivArea'] * 80
    + df['OverallQual'] * 15000
    + df['Bathrooms'] * 8000
    + df['GarageArea'] * 50
    + df['TotalBsmtSF'] * 40
    + np.random.normal(0, 20000, n)
).clip(80000, 600000)

print("Dataset shape:", df.shape)
print("\nFeature value ranges (note the wildly different scales):")
summary = df.describe().loc[['min', 'max', 'mean', 'std']].round(1)
print(summary.to_string())


In [ ]:
# --- Visualization 1: Feature scales before scaling ---
fig, axes = plt.subplots(2, 5, figsize=(18, 8))

feature_cols = [c for c in df.columns if c != 'SalePrice']
colors = plt.cm.tab10.colors

for ax, col in zip(axes.flat, feature_cols):
    ax.hist(df[col], bins=30, color=colors[feature_cols.index(col)], edgecolor='black', alpha=0.8)
    ax.set_title(f'{col}\nRange: [{df[col].min():.0f}, {df[col].max():.0f}]', fontsize=9, fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('Raw Feature Distributions — Notice Vastly Different Scales!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('raw_feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 2: Apply All Scalers


In [ ]:
X = df[feature_cols].copy()
y = df['SalePrice'].copy()

# Apply all 5 scalers
scalers = {
    'Original (No Scaling)': None,
    'StandardScaler':        StandardScaler(),
    'MinMaxScaler':          MinMaxScaler(),
    'RobustScaler':          RobustScaler(),
    'MaxAbsScaler':          MaxAbsScaler(),
}

scaled_data = {}
for name, scaler in scalers.items():
    if scaler is None:
        scaled_data[name] = X.copy()
    else:
        scaled_data[name] = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

# Show the effect on GrLivArea
print("Effect of different scalers on 'GrLivArea':")
print(f"{'Scaler':<30} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}")
print("-" * 62)
for name, df_s in scaled_data.items():
    col = df_s['GrLivArea']
    print(f"{name:<30} {col.min():>10.3f} {col.max():>10.3f} {col.mean():>10.3f} {col.std():>10.3f}")


In [ ]:
# --- Visualization 2: Before vs. After scaling comparison ---
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Select two features with very different scales
cols_to_compare = ['LotArea', 'Bedrooms']

for i, (name, df_s) in enumerate(list(scaled_data.items())[:6]):
    ax = axes.flat[i]
    ax.scatter(df_s['LotArea'], df_s['GrLivArea'], alpha=0.3, s=5,
               color=colors[i])
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('LotArea (after scaling)')
    ax.set_ylabel('GrLivArea (after scaling)')

plt.suptitle('Effect of Different Scalers: LotArea vs GrLivArea', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('scaling_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 3: Impact on Model Performance


In [ ]:
# Compare KNN regression performance with and without scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}
for name, scaler in scalers.items():
    if scaler is None:
        X_train_s = X_train
        X_test_s = X_test
    else:
        # CRITICAL: fit on train only, transform both
        scaler.fit(X_train)
        X_train_s = scaler.transform(X_train)
        X_test_s = scaler.transform(X_test)

    # KNN Regressor (heavily distance-dependent)
    knn = KNeighborsRegressor(n_neighbors=5)
    knn.fit(X_train_s, y_train)
    y_pred = knn.predict(X_test_s)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'KNN RMSE': rmse}

print("KNN Regression RMSE (lower = better):")
print(f"{'Scaler':<30} {'RMSE':>15}")
print("-" * 47)
for name, metrics in results.items():
    print(f"{name:<30} ${metrics['KNN RMSE']:>14,.0f}")

print("\nConclusion: Scaled versions dramatically outperform unscaled for KNN!")


In [ ]:
# Visualize RMSE comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
rmses = [results[n]['KNN RMSE'] for n in names]
bar_colors = ['red' if n == 'Original (No Scaling)' else 'steelblue' for n in names]
bars = ax.bar(names, rmses, color=bar_colors, edgecolor='black')
ax.set_title('KNN Regression RMSE: Impact of Feature Scaling', fontsize=13, fontweight='bold')
ax.set_ylabel('RMSE ($)')
ax.set_xticklabels(names, rotation=20, ha='right')
for bar, rmse in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${rmse:,.0f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('scaling_model_performance.png', dpi=100, bbox_inches='tight')
plt.show()


## 📌 Key Takeaways

- **Always scale** for: KNN, SVM, Logistic Regression, Neural Networks, PCA
- **No need to scale** for: Decision Trees, Random Forest, XGBoost, LightGBM
- **StandardScaler** is your default choice — works well for most algorithms
- **MinMaxScaler** when you need bounded [0,1] range (e.g., image pixels, neural net inputs)
- **RobustScaler** when your data has significant outliers
- **Critical rule**: `fit()` on training data ONLY — then `transform()` on both train and test
- Using `sklearn.Pipeline` automatically handles this correctly
